# **VoxShield - Pipeline Execution Notebook**
This notebook coordinates the complete training and evaluation pipeline for **VoxShield**, a dual-branch attention-based audio deepfake detection system.

### **Important: Choose Your Runtime Environment**

#### **Option A: Running on Local Runtime (Recommended)**
* Use this if you are connecting Google Colab to your **local Jupyter server** (where your GPU and local files are located).
* Do **NOT** run the Google Drive mount code.
* Navigating to your local project directory works out-of-the-box.

#### **Option B: Running on Google Colab Cloud Runtime**
* Use this if you are using Google's hosted cloud GPU.
* Since Google Colab Cloud cannot see your local `D:\` drive directly, you **MUST** first upload your `Dataset` and `VoxShield_Development` folders to your Google Drive.
* Once uploaded, run the Google Drive mount code.

## **Step 1: Environment and Path Setup**
Run the cells below depending on whether you are using a **Local Runtime** or **Cloud Runtime**.

In [ ]:
# RUN THIS ONLY IF USING GOOGLE COLAB CLOUD RUNTIME (Option B)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully.")
except ImportError:
    print("Not running in Colab Cloud. Skipping Drive mount.")

In [2]:
# Install dependencies in your active Python environment
!pip install torch torchaudio transformers soundfile librosa pandas scikit-learn scipy matplotlib

In [3]:
# RUN THIS TO SET UP YOUR PROJECT PATH AND CODE (Supports both Local and Colab Cloud)
import os
if os.path.exists('/content'):
    print('[Environment] Running on Google Colab Cloud.')
    # 1. Check if the code files exist, if not clone from GitHub
    if not os.path.exists('manifest_builder.py'):
        if os.path.exists('VoxShield/VoxShield_Development/manifest_builder.py'):
            get_ipython().run_line_magic('cd', 'VoxShield/VoxShield_Development')
        elif os.path.exists('../manifest_builder.py'):
            get_ipython().run_line_magic('cd', '..')
        else:
            print('Project code files not found in workspace. Cloning from GitHub...')
            get_ipython().system('git clone https://github.com/Sandeepmareeswaran/VoxShield.git')
            if os.path.exists('VoxShield/VoxShield_Development'):
                get_ipython().run_line_magic('cd', 'VoxShield/VoxShield_Development')
            else:
                print('[Warning] Clone finished, but VoxShield_Development directory could not be resolved.')
    
    # 2. Write the 100% correct environment-aware files to the Colab runtime to prevent old/broken GitHub versions
    print('Writing updated environment-aware config.py...')
    config_content = '# config.py\n"""\nVoxShield - Configuration Module\n--------------------------------\nThis module contains the global configuration, hyperparameters, and directory paths.\nIt is shared across all preprocessing, baseline, training, and evaluation scripts.\n\nTo review configurations or import:\n    python -c "import config; print(config.SHARED_DIM)"\n"""\n\nimport os\n\n# Get the directory of config.py (VoxShield_Development)\nBASE_DEV_DIR = os.path.dirname(os.path.abspath(__file__))\n\ndef find_dataset_la():\n    # 1. Check local environment absolute path\n    local_path = r"d:\\Project Work phase 1\\VoxShield\\Dataset\\LA"\n    if os.path.exists(local_path):\n        return local_path\n        \n    # 2. Check relative to code location (e.g., inside Dataset/LA)\n    relative_path_1 = os.path.abspath(os.path.join(BASE_DEV_DIR, "..", "Dataset", "LA"))\n    if os.path.exists(relative_path_1):\n        return relative_path_1\n        \n    # 3. Check if LA is direct sibling of VoxShield_Development\n    relative_path_2 = os.path.abspath(os.path.join(BASE_DEV_DIR, "..", "LA"))\n    if os.path.exists(relative_path_2):\n        return relative_path_2\n        \n    # 4. Search in Google Drive mounts\n    drive_prefixes = [\n        "/content/drive/MyDrive",\n        "/content/drive/Shareddrives",\n    ]\n    for prefix in drive_prefixes:\n        if os.path.exists(prefix):\n            for root, dirs, files in os.walk(prefix):\n                depth = root.replace(prefix, "").count(os.sep)\n                if depth > 3:\n                    continue\n                for d in dirs:\n                    if d.lower() == "la":\n                        la_candidate = os.path.join(root, d)\n                        # Ensure it contains protocols\n                        if os.path.exists(os.path.join(la_candidate, "ASVspoof2019_LA_cm_protocols")):\n                            return la_candidate\n                        # Check also case-insensitive protocol folders\n                        for subd in os.listdir(la_candidate):\n                            if "protocol" in subd.lower():\n                                return la_candidate\n                            \n    return local_path\n\ndef find_dataset_2021():\n    # 1. Check local environment absolute path\n    local_path = r"d:\\Project Work phase 1\\VoxShield\\Dataset\\ASVspoof2021_LA_eval\\ASVspoof2021_LA_eval"\n    if os.path.exists(local_path):\n        return local_path\n        \n    # 2. Check relative to code location (nested ASVspoof2021_LA_eval)\n    relative_path_1 = os.path.abspath(os.path.join(BASE_DEV_DIR, "..", "Dataset", "ASVspoof2021_LA_eval", "ASVspoof2021_LA_eval"))\n    if os.path.exists(relative_path_1):\n        return relative_path_1\n        \n    # 3. Check sibling with or without suffixes\n    parent_dir = os.path.abspath(os.path.join(BASE_DEV_DIR, ".."))\n    dataset_dir = os.path.join(parent_dir, "Dataset")\n    search_dirs = [parent_dir]\n    if os.path.exists(dataset_dir):\n        search_dirs.append(dataset_dir)\n        \n    for s_dir in search_dirs:\n        if os.path.exists(s_dir):\n            for item in os.listdir(s_dir):\n                if "ASVspoof2021_LA_eval" in item:\n                    candidate = os.path.join(s_dir, item)\n                    if os.path.isdir(candidate):\n                        nested_candidate = os.path.join(candidate, "ASVspoof2021_LA_eval")\n                        if os.path.exists(nested_candidate):\n                            return nested_candidate\n                        return candidate\n                    \n    # 4. Search in Google Drive mounts\n    drive_prefixes = [\n        "/content/drive/MyDrive",\n        "/content/drive/Shareddrives",\n    ]\n    for prefix in drive_prefixes:\n        if os.path.exists(prefix):\n            for root, dirs, files in os.walk(prefix):\n                depth = root.replace(prefix, "").count(os.sep)\n                if depth > 3:\n                    continue\n                for d in dirs:\n                    if "ASVspoof2021_LA_eval" in d:\n                        candidate = os.path.join(root, d)\n                        nested_candidate = os.path.join(candidate, "ASVspoof2021_LA_eval")\n                        if os.path.exists(nested_candidate):\n                            return nested_candidate\n                        return candidate\n                        \n    return local_path\n\n# ==============================================================================\n# 1. Directory and Dataset Paths\n# ==============================================================================\n# Absolute paths to raw datasets resolved dynamically\nDATASET_LA_DIR = find_dataset_la()\nDATASET_2021_DIR = find_dataset_2021()\n\n# Path to save generated manifest files and checkpoints\nMANIFEST_DIR = os.path.join(BASE_DEV_DIR, "manifests")\nCHECKPOINT_DIR = os.path.join(BASE_DEV_DIR, "checkpoints")\n\nos.makedirs(MANIFEST_DIR, exist_ok=True)\nos.makedirs(CHECKPOINT_DIR, exist_ok=True)\n\n# Manifest CSV file paths\nTRAIN_2019_CSV = os.path.join(MANIFEST_DIR, "train_2019.csv")\nDEV_2019_CSV = os.path.join(MANIFEST_DIR, "dev_2019.csv")\nEVAL_2019_CSV = os.path.join(MANIFEST_DIR, "eval_2019.csv")\nEVAL_2021_CSV = os.path.join(MANIFEST_DIR, "eval_2021.csv")\n\n# ==============================================================================\n# 2. Audio Preprocessing Parameters\n# ==============================================================================\nSAMPLE_RATE = 16000\nDURATION_SEC = 4.0\nNUM_SAMPLES = int(SAMPLE_RATE * DURATION_SEC)  # 64,000 samples\n\n# ==============================================================================\n# 3. Spectral Feature Parameters (LFCC / Log-Mel)\n# ==============================================================================\n# Options: "lfcc", "mel"\nFEATURE_TYPE = "lfcc"\n\n# LFCC configuration (torchaudio.transforms.LFCC)\nLFCC_PARAMS = {\n    "sample_rate": SAMPLE_RATE,\n    "n_filter": 20,\n    "n_lfcc": 20,\n    "speckwargs": {\n        "n_fft": 512,\n        "win_length": 320,  # 20ms frame length\n        "hop_length": 160,  # 10ms frame shift\n    }\n}\n\n# Log-Mel spectrogram configuration\nMEL_PARAMS = {\n    "sample_rate": SAMPLE_RATE,\n    "n_fft": 512,\n    "win_length": 320,\n    "hop_length": 160,\n    "n_mels": 80\n}\n\n# ==============================================================================\n# 4. Model Architecture Hyperparameters\n# ==============================================================================\n# Model dimensions\nSHARED_DIM = 128            # Shared dimension D (projection target)\nWAV2VEC_MODEL = "facebook/wav2vec2-base"\n\n# CNN Front-end configuration\nCNN_NUM_CHANNELS = [16, 32]  # Convolutional channel growth\nCNN_KERNEL_SIZE = 3\nCNN_POOL_SIZE = 2\n\n# Transformer block configuration\nTRANSFORMER_LAYERS = 2\nTRANSFORMER_HEADS = 4\nTRANSFORMER_FF_DIM = 256\nDROPOUT = 0.1\n\n# Classification parameters\nNUM_CLASSES = 2             # [bonafide, spoof]\n\n# ==============================================================================\n# 5. Training Strategy parameters\n# ==============================================================================\nSEED = 42\nBATCH_SIZE = 32\nEPOCHS = 12                 # Total training epochs (e.g. Stage 1 + Stage 2)\nLR = 1e-4                   # General learning rate for new parameters\nWEIGHT_DECAY = 1e-4\n\n# Frozen vs Fine-tuning stages\nFREEZE_EPOCHS = 6           # Stage 1: Keep wav2vec2 entirely frozen for N epochs\nWAV2VEC_LR = 1e-5           # Stage 2: Unfreeze top layers of wav2vec2 with smaller LR\nUNFREEZE_LAYERS = 2         # Number of top transformer layers of wav2vec2 to unfreeze in Stage 2\n'
    with open('config.py', 'w', encoding='utf-8') as f:
        f.write(config_content)
        
    print('Writing updated environment-aware manifest_builder.py...')
    manifest_content = '# manifest_builder.py\n"""\nVoxShield - Manifest Builder Module (Phase 2)\n---------------------------------------------\nThis script parses the raw protocol/keys files of ASVspoof 2019 LA and ASVspoof 2021 LA\nand saves unified manifest CSV files. If the ASVspoof 2021 keys are missing, it\nautomatically downloads them from the official website.\n\nTo execute this script:\n    python manifest_builder.py\n\nCommand-line usage context (for Colab / Local terminal):\n    !python manifest_builder.py\n"""\n\nimport os\nimport urllib.request\nimport tarfile\nimport ssl\nimport pandas as pd\nimport config\n\n# Disable SSL verification for urllib (necessary for some local environments)\nssl._create_default_https_context = ssl._create_unverified_context\n\ndef find_subfolder(parent, prefix):\n    if not os.path.exists(parent):\n        return os.path.join(parent, prefix)\n    for d in os.listdir(parent):\n        if prefix.lower() in d.lower() and os.path.isdir(os.path.join(parent, d)):\n            return os.path.join(parent, d)\n    return os.path.join(parent, prefix)\n\ndef download_and_extract_2021_keys():\n    """Downloads and extracts the official 2021 evaluation keys if they are not present."""\n    # On Google Colab, use a local fast directory to avoid Google Drive FUSE write slowness/hangs\n    if os.path.exists(\'/content\'):\n        target_dir = "/content/temp_keys"\n    else:\n        target_dir = config.DATASET_2021_DIR\n        \n    os.makedirs(target_dir, exist_ok=True)\n    \n    # Try recursively searching in target_dir or the config dataset directory\n    metadata_path = None\n    for search_dir in [target_dir, config.DATASET_2021_DIR]:\n        if os.path.exists(search_dir):\n            for root, dirs, files in os.walk(search_dir):\n                for f in files:\n                    if "trial_metadata" in f and f.endswith(".txt"):\n                        full_p = os.path.join(root, f)\n                        normalized_p = full_p.replace(\'\\\\\', \'/\')\n                        if "la/cm" in normalized_p.lower():\n                            print(f"[Info] Found ASVspoof 2021 keys at: {full_p}")\n                            return full_p\n                            \n    tar_path = os.path.join(target_dir, "LA-keys-full.tar.gz")\n    if not os.path.exists(tar_path):\n        url = "https://www.asvspoof.org/asvspoof2021/LA-keys-full.tar.gz"\n        print(f"[Info] Downloading ASVspoof 2021 keys from: {url}")\n        try:\n            import socket\n            socket.setdefaulttimeout(30)\n            urllib.request.urlretrieve(url, tar_path)\n            print("[Info] Download complete via urllib.")\n        except Exception as e:\n            print(f"[Warning] Download failed via urllib ({e}). Trying wget fallback...")\n            os.system(f"wget -q -O \\"{tar_path}\\" {url}")\n            print("[Info] Download complete via wget.")\n        \n    print(f"[Info] Extracting keys from {tar_path}...")\n    with tarfile.open(tar_path, "r:gz") as tar:\n        tar.extractall(path=target_dir)\n    print("[Info] Extraction complete.")\n    \n    # Check if we can locate the metadata file now, prioritizing CM\n    candidates_after = []\n    for root, dirs, files in os.walk(target_dir):\n        for f in files:\n            if "trial_metadata" in f and f.endswith(".txt"):\n                full_p = os.path.join(root, f)\n                normalized_p = full_p.replace(\'\\\\\', \'/\')\n                if "la/cm" in normalized_p.lower():\n                    return full_p\n                candidates_after.append(full_p)\n                \n    if candidates_after:\n        return candidates_after[0]\n            \n    raise FileNotFoundError("Could not find trial_metadata.txt in the extracted archive.")\n\ndef build_2019_manifests():\n    """Parses ASVspoof 2019 train, dev, and eval protocols and saves manifest CSVs."""\n    protocols_dir = find_subfolder(config.DATASET_LA_DIR, "ASVspoof2019_LA_cm_protocols")\n    \n    splits = {\n        "train": {\n            "protocol_file": "ASVspoof2019.LA.cm.train.trn.txt",\n            "audio_subfolder": "ASVspoof2019_LA_train",\n            "output_csv": config.TRAIN_2019_CSV\n        },\n        "dev": {\n            "protocol_file": "ASVspoof2019.LA.cm.dev.trl.txt",\n            "audio_subfolder": "ASVspoof2019_LA_dev",\n            "output_csv": config.DEV_2019_CSV\n        },\n        "eval": {\n            "protocol_file": "ASVspoof2019.LA.cm.eval.trl.txt",\n            "audio_subfolder": "ASVspoof2019_LA_eval",\n            "output_csv": config.EVAL_2019_CSV\n        }\n    }\n    \n    for split_name, info in splits.items():\n        # Find exact protocol filename (e.g. if it has a (1) or similar suffix)\n        proto_file_actual = info["protocol_file"]\n        if os.path.exists(protocols_dir):\n            base_prefix = os.path.splitext(info["protocol_file"])[0]\n            for f in os.listdir(protocols_dir):\n                if base_prefix.lower() in f.lower() and f.endswith(".txt"):\n                    proto_file_actual = f\n                    break\n                    \n        proto_path = os.path.join(protocols_dir, proto_file_actual)\n        if not os.path.exists(proto_path):\n            print(f"[Warning] Protocol file {proto_path} not found. Skipping 2019 {split_name} split.")\n            continue\n            \n        print(f"[Info] Building manifest for 2019 {split_name}...")\n        records = []\n        with open(proto_path, "r", encoding="utf-8") as f:\n            for line in f:\n                parts = line.strip().split()\n                if len(parts) >= 5:\n                    speaker_id = parts[0]\n                    utt_id = parts[1]\n                    attack_id = parts[3]\n                    label = parts[4]  # \'bonafide\' or \'spoof\'\n                    \n                    # Construct expected audio file path dynamically to tolerate suffix names\n                    split_folder = find_subfolder(config.DATASET_LA_DIR, info["audio_subfolder"])\n                    flac_folder = find_subfolder(split_folder, "flac")\n                    filepath = os.path.join(flac_folder, f"{utt_id}.flac")\n                    \n                    records.append({\n                        "utt_id": utt_id,\n                        "filepath": filepath,\n                        "label": label,\n                        "split": split_name,\n                        "dataset": "ASVspoof2019_LA",\n                        "attack_id": attack_id if attack_id != "-" else "bonafide",\n                        "speaker_id": speaker_id\n                    })\n                    \n        df = pd.DataFrame(records)\n        df.to_csv(info["output_csv"], index=False)\n        print(f"[Success] Saved {len(df)} records to: {info[\'output_csv\']}")\n\ndef build_2021_manifest():\n    """Parses ASVspoof 2021 evaluation keys and saves eval manifest CSV."""\n    trl_file = None\n    if os.path.exists(config.DATASET_2021_DIR):\n        for f in os.listdir(config.DATASET_2021_DIR):\n            if "ASVspoof2021.LA.cm.eval.trl" in f and f.endswith(".txt"):\n                trl_file = os.path.join(config.DATASET_2021_DIR, f)\n                break\n                \n    if not trl_file:\n        trl_file = os.path.join(config.DATASET_2021_DIR, "ASVspoof2021.LA.cm.eval.trl.txt")\n        \n    if not os.path.exists(trl_file):\n        print(f"[Warning] 2021 trial list {trl_file} not found. Skipping 2021 split.")\n        return\n        \n    try:\n        keys_path = download_and_extract_2021_keys()\n    except Exception as e:\n        print(f"[Error] Failed to get 2021 keys: {e}")\n        return\n        \n    print("[Info] Parsing 2021 evaluation keys metadata...")\n    # trial_metadata columns: [speaker, trial, codec, trans, attack, label, trim, subset]\n    meta_df = pd.read_csv(keys_path, sep=" ", header=None, \n                          names=["speaker_id", "utt_id", "codec", "trans", "attack_id", "label", "trim", "subset"])\n    \n    # Read the trial files that are actually in the evaluation protocol\n    with open(trl_file, "r") as f:\n        eval_trials = set(line.strip() for line in f if line.strip())\n        \n    print(f"[Info] Found {len(eval_trials)} evaluation trial IDs in protocol.")\n    \n    # Filter metadata to keep only relevant evaluation trials\n    filtered_df = meta_df[meta_df["utt_id"].isin(eval_trials)].copy()\n    \n    # Find exact flac folder (e.g. flac or flac (1))\n    flac_dir_name = "flac"\n    if os.path.exists(config.DATASET_2021_DIR):\n        for d in os.listdir(config.DATASET_2021_DIR):\n            if d.lower().startswith("flac") and os.path.isdir(os.path.join(config.DATASET_2021_DIR, d)):\n                flac_dir_name = d\n                break\n                \n    # Construct expected file paths\n    filtered_df["filepath"] = filtered_df["utt_id"].apply(\n        lambda x: os.path.join(config.DATASET_2021_DIR, flac_dir_name, f"{x}.flac")\n    )\n    filtered_df["split"] = "eval"\n    filtered_df["dataset"] = "ASVspoof2021_LA"\n    \n    # Select columns to match the 2019 manifest structure\n    columns_to_keep = ["utt_id", "filepath", "label", "split", "dataset", "attack_id", "speaker_id"]\n    final_df = filtered_df[columns_to_keep]\n    \n    final_df.to_csv(config.EVAL_2021_CSV, index=False)\n    print(f"[Success] Saved {len(final_df)} records to: {config.EVAL_2021_CSV}")\n\nif __name__ == "__main__":\n    print("======================================================================")\n    print("VoxShield - Generating CSV Manifests")\n    print("======================================================================")\n    build_2019_manifests()\n    build_2021_manifest()\n    print("======================================================================")\n    print("All Manifest Generation Complete!")\n    print("======================================================================")\n'
    with open('manifest_builder.py', 'w', encoding='utf-8') as f:
        f.write(manifest_content)
        
    print('Writing updated environment-aware dataset.py...')
    dataset_content = '# dataset.py\n"""\nVoxShield - Dataset Loader Module (Phase 3)\n-------------------------------------------\nThis module implements the PyTorch Dataset for loading ASVspoof audio waveforms,\nhandling resampling to 16 kHz, amplitude normalization, and repeat-padding/cropping\nto exactly 4 seconds (64,000 samples). It returns both raw waveforms and spectral features.\n\nTo run a dataset sanity test:\n    python -c "import dataset; dataset.run_sanity_check()"\n\nCommand-line usage context:\n    Implemented as a library module. Imported into training and evaluation scripts.\n"""\n\nimport os\nimport pandas as pd\nimport torch\nimport torchaudio\nfrom torch.utils.data import Dataset, DataLoader\nimport config\nimport features\n\nclass ASVspoofDataset(Dataset):\n    def __init__(self, manifest_csv, feature_type=None, pad_mode="repeat", return_feats=True):\n        """\n        Args:\n            manifest_csv: Path to the manifest CSV file generated by manifest_builder.py\n            feature_type: Override config.FEATURE_TYPE ("lfcc" or "mel")\n            pad_mode: "repeat" (repeat audio to reach 4s) or "zero" (zero-padding)\n            return_feats: Whether to return extracted features alongside raw waveform\n        """\n        if not os.path.exists(manifest_csv):\n            raise FileNotFoundError(f"Manifest file not found: {manifest_csv}. Run manifest_builder.py first.")\n            \n        self.df = pd.read_csv(manifest_csv)\n        self.feature_type = feature_type if feature_type else config.FEATURE_TYPE\n        self.pad_mode = pad_mode.lower()\n        self.target_length = config.NUM_SAMPLES  # 64,000 samples\n        self.return_feats = return_feats\n\n        # Map labels to integers: bonafide -> 0, spoof -> 1\n        self.label_map = {"bonafide": 0, "spoof": 1}\n        print(f"[Dataset] Loaded {len(self.df)} samples (return_feats={return_feats}) from: {manifest_csv}")\n\n    def __len__(self):\n        return len(self.df)\n\n    def _process_audio(self, filepath):\n        """Loads, resamples, normalizes, and crops/pads the waveform."""\n        # Load waveform with fallback if backend is missing\n        try:\n            waveform, sr = torchaudio.load(filepath)\n        except Exception as e:\n            try:\n                import soundfile as sf\n                data, sr = sf.read(filepath)\n                # If stereo/multichannel, soundfile returns shape [T_samples, Channels]\n                # transpose/reshape to [Channels, T_samples]\n                if len(data.shape) > 1:\n                    # transpose\n                    data = data.T\n                    # take mean if multichannel\n                    waveform = torch.tensor(data, dtype=torch.float32)\n                else:\n                    waveform = torch.tensor(data, dtype=torch.float32).unsqueeze(0)\n            except Exception as sf_err:\n                try:\n                    import librosa\n                    data, sr = librosa.load(filepath, sr=None)\n                    waveform = torch.tensor(data, dtype=torch.float32).unsqueeze(0)\n                except Exception as lib_err:\n                    raise e\n        \n        # Convert stereo/multichannel to mono\n        if waveform.shape[0] > 1:\n            waveform = torch.mean(waveform, dim=0, keepdim=True)\n            \n        # Resample to 16 kHz if necessary\n        if sr != config.SAMPLE_RATE:\n            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=config.SAMPLE_RATE)\n            waveform = resampler(waveform)\n            \n        # Remove channel dimension for processing: [T_samples]\n        waveform = waveform.squeeze(0)\n        \n        # Crop or Pad to target length (64,000 samples = 4 seconds)\n        num_samples = waveform.shape[0]\n        if num_samples >= self.target_length:\n            # Slicing: take the first 4 seconds\n            waveform = waveform[:self.target_length]\n        else:\n            # Padding\n            num_pad = self.target_length - num_samples\n            if self.pad_mode == "repeat":\n                # Repeat the clip until it is long enough\n                repeats = (self.target_length // num_samples) + 1\n                waveform = waveform.repeat(repeats)[:self.target_length]\n            else:\n                # Zero pad\n                waveform = torch.cat([waveform, torch.zeros(num_pad)], dim=0)\n                \n        # Normalization: Zero-mean, Unit-variance (standard for wav2vec2 inputs)\n        mean = waveform.mean()\n        std = waveform.std()\n        waveform = (waveform - mean) / (std + 1e-9)\n        \n        return waveform\n\n    def _translate_filepath(self, filepath, row):\n        """Translates the manifest file path to the current environment path if needed."""\n        # Clean path separators\n        filepath = filepath.replace(\'\\\\\', \'/\')\n        \n        # If the file exists directly, use it\n        if os.path.exists(filepath):\n            return filepath\n            \n        # Get dataset and split details\n        dataset = row.get("dataset", "ASVspoof2019_LA")\n        split = row.get("split", "train")\n        utt_id = row.get("utt_id")\n        \n        # Re-resolve path based on the dataset\n        if dataset == "ASVspoof2021_LA":\n            # For 2021 eval, the audio folder is flac (or flac (1)) inside DATASET_2021_DIR\n            flac_dir_name = "flac"\n            if os.path.exists(config.DATASET_2021_DIR):\n                for d in os.listdir(config.DATASET_2021_DIR):\n                    if d.lower().startswith("flac") and os.path.isdir(os.path.join(config.DATASET_2021_DIR, d)):\n                        flac_dir_name = d\n                        break\n            new_path = os.path.join(config.DATASET_2021_DIR, flac_dir_name, f"{utt_id}.flac")\n            if os.path.exists(new_path):\n                return new_path\n                \n        else:\n            # For 2019 train, dev, eval\n            audio_subfolder = f"ASVspoof2019_LA_{split}"\n            split_folder = self._find_subfolder(config.DATASET_LA_DIR, audio_subfolder)\n            flac_folder = self._find_subfolder(split_folder, "flac")\n            new_path = os.path.join(flac_folder, f"{utt_id}.flac")\n            if os.path.exists(new_path):\n                return new_path\n                \n        # If re-resolution failed but we modified the separator, try it as fallback\n        return filepath\n\n    def _find_subfolder(self, parent, prefix):\n        if not os.path.exists(parent):\n            return os.path.join(parent, prefix)\n        for d in os.listdir(parent):\n            if prefix.lower() in d.lower() and os.path.isdir(os.path.join(parent, d)):\n                return os.path.join(parent, d)\n        return os.path.join(parent, prefix)\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n        filepath = row["filepath"]\n        utt_id = row["utt_id"]\n        label_str = row["label"]\n        label = self.label_map[label_str]\n        \n        # Dynamically translate the filepath if running on Colab or if the path is mismatched\n        filepath = self._translate_filepath(filepath, row)\n        \n        # Load and preprocess raw waveform\n        try:\n            waveform = self._process_audio(filepath)\n        except Exception as e:\n            # Fallback if audio file is corrupted or missing\n            print(f"[Warning] Failed to load {filepath}: {e}. Returning zeros.")\n            waveform = torch.zeros(self.target_length)\n            \n        # Extract features (LFCC or Log-Mel) from the processed waveform if requested\n        # Feature shape output: [1, Features, Frames]\n        if self.return_feats:\n            feats = features.extract_features(waveform, feature_type=self.feature_type)\n        else:\n            feats = torch.empty(0) # return empty tensor to avoid CPU extraction\n            \n        return waveform, feats, label, utt_id\n\ndef run_sanity_check():\n    """Performs dataset validation, checks class balance and feature shapes."""\n    print("=== Running Dataset Sanity Check ===")\n    \n    # We will test using dev manifest first if available\n    manifest_to_check = config.DEV_2019_CSV\n    if not os.path.exists(manifest_to_check):\n        manifest_to_check = config.TRAIN_2019_CSV\n        \n    if not os.path.exists(manifest_to_check):\n        print(f"[Error] No manifest files found at: {manifest_to_check}")\n        print("Please run manifest_builder.py first.")\n        return\n        \n    dataset = ASVspoofDataset(manifest_to_check)\n    \n    # Check balance\n    counts = dataset.df["label"].value_counts()\n    print(f"Class Balance in manifest:\\n{counts}")\n    \n    # Check stats\n    lengths = []\n    print("Checking first 5 files...")\n    for i in range(min(5, len(dataset))):\n        waveform, feats, label, utt_id = dataset[i]\n        print(f"Sample {i} | ID: {utt_id} | Label: {label} | Waveform: {waveform.shape} | Features: {feats.shape}")\n        \n    print("=== Sanity Check Complete ===")\n\nif __name__ == "__main__":\n    run_sanity_check()\n'
    with open('dataset.py', 'w', encoding='utf-8') as f:
        f.write(dataset_content)
        
    print('Files patched successfully!')
    
    # 3. Copy dataset to local SSD (/content/VoxShield/LA) to eliminate Google Drive FUSE mount slowness
    local_la = '/content/VoxShield/LA'
    drive_la = '/content/drive/MyDrive/project phase 1/LA'
    if not os.path.exists(drive_la):
        drive_la = '/content/drive/MyDrive/Project Work phase 1/Dataset/LA'
        
    # Search for compressed archives in Drive to bypass FUSE metadata overhead
    parent_dir = os.path.dirname(drive_la) if os.path.exists(os.path.dirname(drive_la)) else ''
    zip_path = os.path.join(parent_dir, 'LA.zip') if parent_dir else ''
    tar_path = os.path.join(parent_dir, 'LA.tar.gz') if parent_dir else ''
    
    if not os.path.exists(local_la):
        if zip_path and os.path.exists(zip_path):
            print(f'Found zipped dataset at: {zip_path}')
            print('Unzipping dataset directly to local SSD (extremely fast, ~1-2 mins)...')
            os.makedirs(local_la, exist_ok=True)
            get_ipython().system(f"unzip -qo '{zip_path}' -d '{local_la}'")
            print('Unzip complete!')
        elif tar_path and os.path.exists(tar_path):
            print(f'Found tarball dataset at: {tar_path}')
            print('Extracting tarball directly to local SSD (extremely fast, ~1-2 mins)...')
            os.makedirs(local_la, exist_ok=True)
            get_ipython().system(f"tar -zxf '{tar_path}' -C '{local_la}'")
            print('Extraction complete!')
        elif os.path.exists(drive_la):
            print(f'Dataset folder found at: {drive_la}')
            print('Copying dataset files in parallel to local Colab SSD to bypass FUSE latency (max 32 threads)...')
            os.makedirs(local_la, exist_ok=True)
            
            import shutil
            import time
            from concurrent.futures import ThreadPoolExecutor
            
            def copy_single_file(src_dst):
                src, dst = src_dst
                try:
                    shutil.copy2(src, dst)
                except Exception:
                    pass
            
            print('Scanning Google Drive folder structure (please wait)...')
            files_to_copy = []
            for root, dirs, files in os.walk(drive_la):
                rel_path = os.path.relpath(root, drive_la)
                target_dir = os.path.join(local_la, rel_path) if rel_path != '.' else local_la
                os.makedirs(target_dir, exist_ok=True)
                for f in files:
                    files_to_copy.append((os.path.join(root, f), os.path.join(target_dir, f)))
            
            print(f'Copying {len(files_to_copy)} files in parallel (this runs much faster than tar)...')
            start_time = time.time()
            with ThreadPoolExecutor(max_workers=32) as executor:
                list(executor.map(copy_single_file, files_to_copy))
            print(f'Dataset copy complete in {time.time() - start_time:.2f} seconds!')
        else:
            print('[Warning] LA dataset path not found. Proceeding without local copy.')
       
    
    # 4. Dynamically locate and navigate to the project directory in Google Drive if mounted
    # (Only if we didn't just clone the code repo into /content)
    if not os.path.exists('manifest_builder.py'):
        def find_and_cd():
            candidates = [
                '/content/drive/MyDrive/project phase 1/VoxShield/VoxShield_Development',
                '/content/drive/MyDrive/Project Work phase 1/VoxShield/VoxShield_Development',
                '/content/drive/MyDrive/VoxShield/VoxShield_Development',
                '/content/drive/MyDrive/project phase 1/VoxShield_Development',
            ]
            for c in candidates:
                if os.path.exists(c):
                    print(f'Found project development folder at: {c}')
                    get_ipython().run_line_magic('cd', c)
                    return True
            print('Searching for VoxShield_Development in Google Drive...')
            for root, dirs, files in os.walk('/content/drive/MyDrive'):
                depth = root.replace('/content/drive/MyDrive', '').count(os.sep)
                if depth > 3: continue
                if 'VoxShield_Development' in dirs:
                    target = os.path.join(root, 'VoxShield_Development')
                    print(f'Found project development folder at: {target}')
                    get_ipython().run_line_magic('cd', target)
                    return True
            print('[Error] Could not automatically find VoxShield_Development in Google Drive.')
            print('Please run manually: %cd "/content/drive/MyDrive/your-path/VoxShield_Development"')
            return False
        find_and_cd()
    else:
        print(f'Active working directory containing code files: {os.getcwd()}')
else:
    print('[Environment] Running on Local Machine.')
    local_path = 'D:/Project Work phase 1/VoxShield/VoxShield_Development'
    if os.path.exists(local_path):
        get_ipython().run_line_magic('cd', local_path)
        print(f'Navigated to local project folder: {local_path}')
    else:
        print(f'[Warning] Local path not found: {local_path}')

if os.path.exists('/content'):
    get_ipython().system('ls')
else:
    get_ipython().system('dir')


[Environment] Running on Google Colab Cloud.
Project code files not found in workspace. Cloning from GitHub...
Cloning into 'VoxShield'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 116 (delta 47), reused 81 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 17.01 MiB | 17.18 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/VoxShield/VoxShield_Development
Writing updated environment-aware config.py...
Writing updated environment-aware manifest_builder.py...
Writing updated environment-aware dataset.py...
Files patched successfully!
Dataset folder found at: /content/drive/MyDrive/project phase 1/LA
Copying dataset files in parallel to local Colab SSD to bypass FUSE latency (max 32 threads)...
Scanning Google Drive folder structure (please wait)...
Copying 50401 files in parallel (this runs much faster than tar)...
Dataset copy complete in 843.41 secon

## **Step 2: Generate Dataset Manifests (Phase 2)**
This script parses the protocol files for ASVspoof 2019 and ASVspoof 2021 logical access trials and writes standard CSV manifest lists.

In [4]:
# Run manifest builder script
!python manifest_builder.py

VoxShield - Generating CSV Manifests
[Info] Building manifest for 2019 train...
[Success] Saved 25380 records to: /content/VoxShield/VoxShield_Development/manifests/train_2019.csv
[Info] Building manifest for 2019 dev...
[Success] Saved 24844 records to: /content/VoxShield/VoxShield_Development/manifests/dev_2019.csv
[Info] Building manifest for 2019 eval...
[Success] Saved 71237 records to: /content/VoxShield/VoxShield_Development/manifests/eval_2019.csv
[Info] Found ASVspoof 2021 keys at: /content/drive/MyDrive/project phase 1/ASVspoof2021_LA_eval (1)/keys/LA/CM/trial_metadata.txt
[Info] Parsing 2021 evaluation keys metadata...
[Info] Found 181566 evaluation trial IDs in protocol.
[Success] Saved 181566 records to: /content/VoxShield/VoxShield_Development/manifests/eval_2021.csv
All Manifest Generation Complete!


## **Step 3: Preprocessing and Dataloader Sanity Checks (Phase 3 & 4)**
Verify the dataset class loading, resampling, repeat padding, and default feature extraction (LFCC) shapes.

In [5]:
!python dataset.py

=== Running Dataset Sanity Check ===
[Dataset] Loaded 24844 samples (return_feats=True) from: /content/VoxShield/VoxShield_Development/manifests/dev_2019.csv
Class Balance in manifest:
label
spoof       22296
bonafide     2548
Name: count, dtype: int64
Checking first 5 files...
Sample 0 | ID: LA_D_1047731 | Label: 0 | Waveform: torch.Size([64000]) | Features: torch.Size([1, 20, 401])
Sample 1 | ID: LA_D_1105538 | Label: 0 | Waveform: torch.Size([64000]) | Features: torch.Size([1, 20, 401])
Sample 2 | ID: LA_D_1125976 | Label: 0 | Waveform: torch.Size([64000]) | Features: torch.Size([1, 20, 401])
Sample 3 | ID: LA_D_1293230 | Label: 0 | Waveform: torch.Size([64000]) | Features: torch.Size([1, 20, 401])
Sample 4 | ID: LA_D_1340209 | Label: 0 | Waveform: torch.Size([64000]) | Features: torch.Size([1, 20, 401])
=== Sanity Check Complete ===


## **Step 4: Train and Evaluate Baseline Models (Phase 5)**
Train both frozen wav2vec2-base (Baseline A) and LFCC-CNN (Baseline B) classifier pipelines to establish comparison results.

In [ ]:
# Train Baseline A (wav2vec2 + MLP classifier)
!python baselines/baseline_wav2vec2.py

[Info] Using device: cuda
[Info] Loading datasets...
[Dataset] Loaded 25380 samples (return_feats=False) from: /content/VoxShield/VoxShield_Development/manifests/train_2019.csv
[Dataset] Loaded 24844 samples (return_feats=False) from: /content/VoxShield/VoxShield_Development/manifests/dev_2019.csv
[Info] Initializing Baseline A model (Frozen wav2vec2)...
config.json: 100% 1.84k/1.84k [00:00<00:00, 5.04MB/s]
pytorch_model.bin: 100% 380M/380M [00:08<00:00, 46.5MB/s] 
Loading weights: 100% 211/211 [00:00<00:00, 29252.27it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight

In [ ]:
# Train Baseline B (LFCC + 2D CNN classifier)
!python baselines/baseline_cnn.py

## **Step 5: Train VoxShield Core Architecture (Phase 6)**
This script trains the twin-branch fused VoxShield architecture using a two-stage scheduler (initially frozen wav2vec2, followed by unfreezing of the top 2 layers with a low learning rate).

In [ ]:
!python train.py

## **Step 6: Logit Calibration (Phase 7)**
Compute optimal temperature scaling on the validation dataset (dev_2019) to calibrate probability outputs.

In [ ]:
!python calibrate.py

## **Step 7: Inference / Prediction (Phase 7)**
Run prediction on any target flac audio file. The script outputs the calibrated spoof probability, the associated risk band (Green, Amber, Red), and highlights the most suspicious segment timeframes.

In [ ]:
# Run inference on a sample dev file (update filepath to test other files)
!python predict.py --audio_path "../Dataset/LA/ASVspoof2019_LA_dev/flac/LA_D_1048473.flac"

## **Step 8: Export Trained Models to Google Drive**
This step copies the trained checkpoints from the local Colab runtime to your mounted Google Drive for persistent storage, and outputs the destination path where they are located.

In [ ]:
# Import required libraries
import os
import shutil
try:
    import config
    checkpoint_dir = config.CHECKPOINT_DIR
except Exception:
    checkpoint_dir = './checkpoints'

# Define Google Drive destination path
drive_target_dir = '/content/drive/MyDrive/VoxShield_Checkpoints'

if os.path.exists('/content'):
    # Ensure Google Drive is mounted
    if not os.path.exists('/content/drive'):
        print("Google Drive is not mounted. Attempting to mount...")
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except ImportError:
            print("[Error] Failed to import google.colab.drive. Cannot automatically mount.")
            
    if os.path.exists('/content/drive'):
        os.makedirs(drive_target_dir, exist_ok=True)
        
        # Copy all checkpoints if the directory exists and has files
        if os.path.exists(checkpoint_dir) and os.listdir(checkpoint_dir):
            print(f"Copying trained checkpoints from {os.path.abspath(checkpoint_dir)} to Google Drive...")
            copied_files = []
            for file_name in os.listdir(checkpoint_dir):
                src_file = os.path.join(checkpoint_dir, file_name)
                dst_file = os.path.join(drive_target_dir, file_name)
                if os.path.isfile(src_file):
                    shutil.copy2(src_file, dst_file)
                    copied_files.append(file_name)
            print(f"[Success] Copied checkpoints: {copied_files}")
            print(f"Trained model checkpoints are stored in Google Drive at: {drive_target_dir}")
        else:
            print(f"[Warning] No checkpoints found in '{checkpoint_dir}'. Please ensure training has run successfully.")
    else:
        print("[Error] Google Drive is not mounted. Please mount Google Drive to save the trained model.")
else:
    print(f"[Environment] Running on Local Machine. Checkpoints are already saved at: {os.path.abspath(checkpoint_dir)}")
